# Scraper for 2025 H2H Shoots

Run the code cell below to import all rows in the **Shoots** sheet where **Imported = 0**.

In [13]:
from pathlib import Path
import re
import sys

import pandas as pd
from openpyxl import load_workbook

# Resolve paths whether kernel cwd is repo root or the notebook folder.
CWD = Path.cwd()
if (CWD / "main.py").exists() and (CWD / "2025 H2Hs").exists():
    PROJECT_ROOT = CWD
    NOTEBOOK_DIR = CWD / "2025 H2Hs"
else:
    NOTEBOOK_DIR = CWD
    PROJECT_ROOT = CWD.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from main import _clean_scraped_table  # Reuse existing table-cleaning logic.

WORKBOOK_PATH = NOTEBOOK_DIR / "2025 H2H Shoots.xlsx"
if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(f"Workbook not found: {WORKBOOK_PATH}")
SHOOTS_SHEET = "Shoots"


def _find_column(columns, candidate_names):
    lowered = {str(col).strip().lower(): col for col in columns}
    for name in candidate_names:
        if name.lower() in lowered:
            return lowered[name.lower()]
    raise KeyError(f"Missing required column. Expected one of: {candidate_names}")


def _clean_club(country_value):
    if pd.isna(country_value):
        return pd.NA
    text = str(country_value).strip()
    return re.sub(r"^\s*\d+\s*-\s*", "", text).strip()


def _scrape_event_table(url):
    tables = pd.read_html(url, flavor="lxml")
    if not tables:
        raise ValueError(f"No HTML table found at {url}")

    cleaned = _clean_scraped_table(tables[0])
    club_source_col = _find_column(
        cleaned.columns,
        ["Country", "Country or State Code"],
    )
    required = ["Pos.", "Athlete", "Tot.", club_source_col]
    missing = [col for col in required if col not in cleaned.columns]
    if missing:
        raise KeyError(f"Missing expected columns on page {url}: {missing}")

    result = cleaned.loc[:, required].copy()
    result["Club"] = result[club_source_col].apply(_clean_club)
    result = result.rename(
        columns={
            "Pos.": "Rank",
            "Athlete": "Name",
            "Tot.": "Score",
        }
    )
    return result[["Rank", "Name", "Club", "Score"]]


# Load workbook metadata from Shoots sheet.
shoots_df = pd.read_excel(WORKBOOK_PATH, sheet_name=SHOOTS_SHEET)
url_col = _find_column(shoots_df.columns, ["URL", "Url"])
event_col = _find_column(shoots_df.columns, ["Event Name", "Event"])
tier_col = _find_column(shoots_df.columns, ["Tier"])
imported_col = _find_column(shoots_df.columns, ["Imported", "imported"])
ranking_points_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Ranking Points")
ranking_rank_col = _find_column(ranking_points_df.columns, ["Rank", "Pos.", "Position"])

ranking_points_df[ranking_rank_col] = pd.to_numeric(
    ranking_points_df[ranking_rank_col],
    errors="coerce",
)
ranking_points_df = ranking_points_df[ranking_points_df[ranking_rank_col].notna()].copy()


def _normalise_col_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def _points_col_for_tier(columns, tier_value):
    tier_text = str(tier_value).strip()
    tier_num = tier_text
    if tier_text.replace(".", "", 1).isdigit():
        tier_num = str(int(float(tier_text)))

    candidates = [
        tier_text,
        tier_num,
        f"Tier {tier_text}",
        f"Tier {tier_num}",
        f"Tier {tier_text} Points",
        f"Tier {tier_num} Points",
        f"Tier{tier_text}Points",
        f"Tier{tier_num}Points",
        f"T{tier_text}",
        f"T{tier_num}",
    ]

    normalised = {_normalise_col_name(c): c for c in columns}
    for candidate in candidates:
        key = _normalise_col_name(candidate)
        if key in normalised:
            return normalised[key]

    for col in columns:
        key = _normalise_col_name(col)
        if "tier" in key and tier_num in key and "point" in key:
            return col

    raise KeyError(f"Could not find ranking-points column for tier {tier_text}")


def _ranking_points_map_for_tier(tier_value):
    points_col = _points_col_for_tier(ranking_points_df.columns, tier_value)
    tmp = ranking_points_df[[ranking_rank_col, points_col]].copy()
    tmp[points_col] = pd.to_numeric(tmp[points_col], errors="coerce")
    tmp = tmp[tmp[points_col].notna()].copy()
    tmp[ranking_rank_col] = tmp[ranking_rank_col].astype(int)
    return {int(r): float(p) for r, p in zip(tmp[ranking_rank_col], tmp[points_col])}


def _apply_ranking_points(event_df, tier_value):
    rank_col = _find_column(event_df.columns, ["Rank", "Pos.", "Position"])
    rank_points = _ranking_points_map_for_tier(tier_value)
    ranks = pd.to_numeric(event_df[rank_col], errors="coerce")

    def _lookup_points(rank_value):
        if pd.isna(rank_value):
            return 0.0
        return float(rank_points.get(int(rank_value), 0.0))

    out = event_df.copy()
    out["Ranking Points"] = ranks.apply(_lookup_points)
    out["Ranking Points"] = pd.to_numeric(out["Ranking Points"], errors="coerce").fillna(0.0)
    return out


event_tier_df = (
    shoots_df[[event_col, tier_col]]
    .dropna(subset=[event_col])
    .assign(**{event_col: lambda d: d[event_col].astype(str).str.strip()})
    .loc[lambda d: d[event_col] != ""]
    .drop_duplicates(subset=[event_col], keep="first")
)
event_tier_map = dict(zip(event_tier_df[event_col], event_tier_df[tier_col]))

pending_mask = shoots_df[imported_col].fillna(0).astype(int) == 0
pending_rows = shoots_df[pending_mask].copy()

wb = load_workbook(WORKBOOK_PATH)
processed = []
ranking_updated = []
errors = []

if pending_rows.empty:
    print("No rows with Imported = 0 were found.")
else:
    for row_index, row in pending_rows.iterrows():
        event_name = str(row[event_col]).strip()
        url = str(row[url_col]).strip()

        if not event_name or event_name.lower() == "nan":
            errors.append(f"Row {row_index}: missing Event Name")
            continue
        if not url or url.lower() == "nan":
            errors.append(f"Row {row_index}: missing URL")
            continue

        try:
            event_df = _scrape_event_table(url)
            event_df = _apply_ranking_points(event_df, row[tier_col])

            if event_name in wb.sheetnames:
                del wb[event_name]
            ws = wb.create_sheet(title=event_name)

            ws.append(list(event_df.columns))
            for values in event_df.itertuples(index=False, name=None):
                ws.append(list(values))

            shoots_df.at[row_index, imported_col] = 1
            processed.append((event_name, len(event_df)))
        except Exception as exc:
            errors.append(f"{event_name}: {exc}")

# Ensure existing event sheets have Ranking Points, independent of URL scraping.
for event_name, tier_value in event_tier_map.items():
    if event_name not in wb.sheetnames:
        continue

    try:
        ws_event = wb[event_name]
        rows = list(ws_event.values)
        if not rows:
            continue

        headers = [str(h).strip() if h is not None else "" for h in rows[0]]
        data_rows = rows[1:]
        existing_df = pd.DataFrame(data_rows, columns=headers)

        needs_points_fill = "Ranking Points" not in existing_df.columns
        if not needs_points_fill:
            points_numeric = pd.to_numeric(existing_df["Ranking Points"], errors="coerce")
            needs_points_fill = points_numeric.isna().any()

        if not needs_points_fill:
            continue

        updated_df = _apply_ranking_points(existing_df, tier_value)

        del wb[event_name]
        ws_new = wb.create_sheet(title=event_name)
        ws_new.append(updated_df.columns.tolist())
        for values in updated_df.itertuples(index=False, name=None):
            ws_new.append(list(values))
        ranking_updated.append(event_name)
    except Exception as exc:
        errors.append(f"{event_name} (ranking points): {exc}")

# Rewrite Shoots sheet with updated Imported flags.
if SHOOTS_SHEET in wb.sheetnames:
    del wb[SHOOTS_SHEET]
ws_shoots = wb.create_sheet(title=SHOOTS_SHEET, index=0)
ws_shoots.append(shoots_df.columns.tolist())
for values in shoots_df.itertuples(index=False, name=None):
    ws_shoots.append(list(values))

wb.save(WORKBOOK_PATH)

print(f"Processed events: {len(processed)}")
for event_name, rows_written in processed:
    print(f"  - {event_name}: {rows_written} rows")

print(f"Ranking Points updated in sheets: {len(ranking_updated)}")
for event_name in sorted(ranking_updated):
    print(f"  - {event_name}")

if errors:
    print("\nEvents with errors:")
    for item in errors:
        print(f"  - {item}")

No rows with Imported = 0 were found.
Processed events: 0
Ranking Points updated in sheets: 4
  - AGB NT S1
  - AGB NT S2
  - AGB NT S3
  - AGB NT S6 (BTC)


# Plotting

In [14]:
from pathlib import Path
import random

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def _find_col(columns, candidate_names):
    lowered = {str(col).strip().lower(): col for col in columns}
    for name in candidate_names:
        if name.lower() in lowered:
            return lowered[name.lower()]
    raise KeyError(f"Missing required column. Expected one of: {candidate_names}")


CWD = Path.cwd()
if (CWD / "2025 H2Hs" / "2025 H2H Shoots.xlsx").exists():
    workbook_path = CWD / "2025 H2Hs" / "2025 H2H Shoots.xlsx"
else:
    workbook_path = CWD / "2025 H2H Shoots.xlsx"

if not workbook_path.exists():
    raise FileNotFoundError(f"Workbook not found: {workbook_path}")

shoots_df = pd.read_excel(workbook_path, sheet_name="Shoots")
event_col = _find_col(shoots_df.columns, ["Event Name", "Event"])
tier_col = _find_col(shoots_df.columns, ["Tier"])

events_tiers = (
    shoots_df[[event_col, tier_col]]
    .dropna(subset=[event_col])
    .assign(
        **{
            event_col: lambda d: d[event_col].astype(str).str.strip(),
            tier_col: lambda d: d[tier_col].astype(str).str.strip(),
        }
    )
)

events_tiers = events_tiers[events_tiers[event_col] != ""]
events_tiers = events_tiers.drop_duplicates(subset=[event_col], keep="first")

xlsx = pd.ExcelFile(workbook_path)
available_sheets = set(xlsx.sheet_names)

tier1_green_palette = [
    "#1b5e20", "#2e7d32", "#388e3c", "#43a047", "#4caf50",
    "#66bb6a", "#81c784", "#2f6f3e", "#3c8d4f", "#5aa469",
]
tier2_coolhot_palette = [
    "#b71c1c", "#d32f2f", "#e53935", "#ad1457", "#c2185b",
    "#8e24aa", "#7b1fa2", "#5e35b1", "#3949ab", "#1e40af",
    "#1565c0", "#1976d2", "#0d47a1", "#6a1b9a", "#880e4f",
]

rng = random.Random(42)
events_by_tier = (
    events_tiers.groupby(tier_col)[event_col]
    .apply(lambda s: [str(v).strip() for v in s.tolist()])
    .to_dict()
)

tier1_events = events_by_tier.get("1", [])
tier1_color_map = {
    event_name: tier1_green_palette[i % len(tier1_green_palette)]
    for i, event_name in enumerate(tier1_events)
}

tier2_events = events_by_tier.get("2", [])
tier2_palette_shuffled = tier2_coolhot_palette[:]
rng.shuffle(tier2_palette_shuffled)
tier2_color_map = {
    event_name: tier2_palette_shuffled[i % len(tier2_palette_shuffled)]
    for i, event_name in enumerate(tier2_events)
}

fig = go.Figure()
plotted = 0
skipped = []

for event_name_raw, tier_raw in events_tiers[[event_col, tier_col]].itertuples(index=False, name=None):
    event_name = str(event_name_raw).strip()
    tier = str(tier_raw).strip()

    if event_name not in available_sheets:
        skipped.append((event_name, "sheet not found"))
        continue

    event_df = pd.read_excel(workbook_path, sheet_name=event_name)

    try:
        rank_col = _find_col(event_df.columns, ["Rank"])
        points_col = _find_col(event_df.columns, ["Ranking Points"])
        score_col = _find_col(event_df.columns, ["Score"])
        name_col = _find_col(event_df.columns, ["Name", "Athlete"])
        club_col = _find_col(event_df.columns, ["Club", "Country", "Country or State Code"])
    except KeyError as exc:
        skipped.append((event_name, str(exc)))
        continue

    plot_df = event_df[[rank_col, points_col, score_col, name_col, club_col]].copy()
    plot_df[rank_col] = pd.to_numeric(plot_df[rank_col], errors="coerce")
    plot_df[points_col] = pd.to_numeric(plot_df[points_col], errors="coerce")
    plot_df[score_col] = pd.to_numeric(plot_df[score_col], errors="coerce")
    plot_df = plot_df.dropna(subset=[rank_col, score_col])

    if plot_df.empty:
        skipped.append((event_name, "no numeric rank/score rows"))
        continue

    plot_df = plot_df.sort_values(rank_col)

    if tier == "1":
        marker_color = tier1_color_map.get(event_name, "#2e7d32")
    elif tier == "2":
        marker_color = tier2_color_map.get(event_name, "#3949ab")
    else:
        marker_color = "#4c78a8"

    fig.add_trace(
        go.Scatter(
            x=plot_df[rank_col],
            y=plot_df[score_col],
            mode="lines+markers",
            name=f"{event_name} (Tier {tier})",
            legendgroup=f"Tier {tier}",
            line={"color": marker_color, "width": 2},
            marker={"color": marker_color, "size": 5, "symbol": "circle"},
            opacity=0.7,
            customdata=plot_df[[name_col, club_col, rank_col, points_col]].astype(str).to_numpy(),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                + "Rank: %{customdata[2]}<br>"
                + "Ranking Points: %{customdata[3]}<br>"
                + "Score: %{y}<br>"
                + "Name: %{customdata[0]}<br>"
                + "Club: %{customdata[1]}<extra></extra>"
            ),
        )
    )
    fig.data[-1].meta = {
        "rank_x": plot_df[rank_col].tolist(),
        "points_x": plot_df[points_col].tolist(),
    }
    plotted += 1

fig.update_layout(
    title="Score vs Rank Across 2025 H2H Events",
    xaxis_title="Rank",
    yaxis_title="Score",
    hovermode="closest",
    template="plotly_white",
    showlegend=False,
    xaxis={"range": [-0.5, 30]},
    yaxis={"range": [500, 680]},
    width=1200,
    height=700,
)

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(0, 31)),
    ticktext=[str(v) if v % 5 == 0 else "" for v in range(0, 31)],
)
fig.add_hline(y=580, line_dash="dash", line_color="#cd7f32", line_width=4)
fig.add_hline(y=600, line_dash="dash", line_color="#c0c0c0", line_width=4)
fig.add_hline(y=620, line_dash="dash", line_color="#ffd700", line_width=4)

fig_widget = go.FigureWidget(fig)

RANK_HOVER = (
    "<b>%{fullData.name}</b><br>"
    + "Rank: %{customdata[2]}<br>"
    + "Ranking Points: %{customdata[3]}<br>"
    + "Score: %{y}<br>"
    + "Name: %{customdata[0]}<br>"
    + "Club: %{customdata[1]}<extra></extra>"
)
POINTS_HOVER = (
    "<b>%{fullData.name}</b><br>"
    + "Ranking Points: %{x}<br>"
    + "Rank: %{customdata[2]}<br>"
    + "Score: %{y}<br>"
    + "Name: %{customdata[0]}<br>"
    + "Club: %{customdata[1]}<extra></extra>"
)

def _switch_x_mode(mode):
    with fig_widget.batch_update():
        if mode == "rank":
            for trace in fig_widget.data:
                trace.x = trace.meta.get("rank_x", [])
                trace.hovertemplate = RANK_HOVER
            fig_widget.update_layout(xaxis_title="Rank")
            fig_widget.update_xaxes(
                range=[-0.5, 30],
                autorange=False,
                tickmode="array",
                tickvals=list(range(0, 31)),
                ticktext=[str(v) if v % 5 == 0 else "" for v in range(0, 31)],
            )
        else:
            for trace in fig_widget.data:
                trace.x = trace.meta.get("points_x", [])
                trace.hovertemplate = POINTS_HOVER
            fig_widget.update_layout(xaxis_title="Ranking Points")
            fig_widget.update_xaxes(
                range=None,
                autorange="reversed",
                tickmode="auto",
                tickvals=None,
                ticktext=None,
            )

x_mode_toggle = widgets.ToggleButtons(
    options=[("Score vs Rank", "rank"), ("Score vs Points", "points")],
    value="rank",
    description="View:",
    style={"description_width": "initial"},
)
x_mode_toggle.observe(
    lambda change: _switch_x_mode(change["new"]) if change["name"] == "value" else None,
    names="value",
)
_switch_x_mode("rank")
event_controls = []
for i, trace in enumerate(fig_widget.data):
    cb = widgets.Checkbox(
        value=True,
        description=trace.name,
        indent=False,
        layout=widgets.Layout(width="334px"),
    )
    swatch = widgets.HTML(
        value=f"<span style='color:{trace.line.color};font-size:16px;line-height:1;'>■</span>",
        layout=widgets.Layout(width="18px"),
    )

    def _make_handler(trace_index):
        def _on_change(change):
            if change["name"] == "value":
                fig_widget.data[trace_index].visible = bool(change["new"])

        return _on_change

    cb.observe(_make_handler(i), names="value")
    event_controls.append((trace.name.lower(), widgets.HBox([swatch, cb])))

event_controls_sorted = [control for _, control in sorted(event_controls, key=lambda t: t[0])]

checkbox_panel = widgets.VBox(
    [
        widgets.HTML("<b>Show / hide events (A-Z)</b>"),
    ] + event_controls_sorted,
    layout=widgets.Layout(
        width="380px",
        max_height="700px",
        overflow_y="auto",
        border="1px solid #d0d0d0",
        padding="8px",
    ),
)
display(
    widgets.VBox(
        [
            widgets.HBox([widgets.HTML("<b>Plot Mode:</b>"), x_mode_toggle]),
            widgets.HBox([checkbox_panel, fig_widget], layout=widgets.Layout(align_items="flex-start")),
        ]
    )
)

print(f"Plotted events: {plotted}")
if skipped:
    print("Skipped events:")
    for event_name, reason in skipped:
        print(f"  - {event_name}: {reason}")

Plotted events: 30


# List Least Competitive Events
- List events whereby 5th place scored less than 620, 600 and 580 respectively

In [15]:
from pathlib import Path

import pandas as pd


def _find_col(columns, candidate_names):
    lowered = {str(col).strip().lower(): col for col in columns}
    for name in candidate_names:
        if name.lower() in lowered:
            return lowered[name.lower()]
    raise KeyError(f"Missing required column. Expected one of: {candidate_names}")


CWD = Path.cwd()
if (CWD / "2025 H2Hs" / "2025 H2H Shoots.xlsx").exists():
    workbook_path = CWD / "2025 H2Hs" / "2025 H2H Shoots.xlsx"
else:
    workbook_path = CWD / "2025 H2H Shoots.xlsx"

if not workbook_path.exists():
    raise FileNotFoundError(f"Workbook not found: {workbook_path}")

shoots_df = pd.read_excel(workbook_path, sheet_name="Shoots")
event_col = _find_col(shoots_df.columns, ["Event Name", "Event"])

events = (
    shoots_df[event_col]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .drop_duplicates()
    .tolist()
)

available_sheets = set(pd.ExcelFile(workbook_path).sheet_names)
thresholds = [620, 600, 580]
classification_thresholds = sorted(thresholds)
below = {t: [] for t in thresholds}
skipped = []

for event_name in events:
    if event_name not in available_sheets:
        skipped.append((event_name, "sheet not found"))
        continue

    df = pd.read_excel(workbook_path, sheet_name=event_name)

    try:
        rank_col = _find_col(df.columns, ["Rank"])
        score_col = _find_col(df.columns, ["Score"])
    except KeyError as exc:
        skipped.append((event_name, str(exc)))
        continue

    work = df[[rank_col, score_col]].copy()
    work[rank_col] = pd.to_numeric(work[rank_col], errors="coerce")
    work[score_col] = pd.to_numeric(work[score_col], errors="coerce")
    work = work.dropna(subset=[rank_col, score_col])

    fifth_row = work.loc[work[rank_col] == 5, score_col]
    if fifth_row.empty:
        skipped.append((event_name, "no rank 5 row"))
        continue

    fifth_score = float(fifth_row.iloc[0])

    bucket = next((t for t in classification_thresholds if fifth_score < t), None)
    if bucket is not None:
        below[bucket].append((event_name, fifth_score))

for t in thresholds:
    print(f"\nEvents where 5th rank score < {t}:")
    rows = sorted(below[t], key=lambda x: x[1])
    if not rows:
        print("  - None")
    else:
        for event_name, score in rows:
            print(f"  - {event_name}: {score:.0f}")

if skipped:
    print("\nSkipped events:")
    for event_name, reason in skipped:
        print(f"  - {event_name}: {reason}")


Events where 5th rank score < 620:
  - Cleve: 606
  - Bowmen of Glen May Sunday: 608
  - Guildford: 609
  - Bowmen of Glen Sep Saturday: 610
  - Meriden Spring Bank Holiday: 618

Events where 5th rank score < 600:
  - Greasley Castle: 583
  - Llantarnam: 587
  - Bowmen of Glen Sep Sunday: 587
  - Meriden Mid-Summer: 588
  - Wallingford: 588
  - Sherwood: 589
  - Kent: 591
  - DPA: 592
  - Eccles: 596
  - Long Mynd: 597
  - Andover: 599

Events where 5th rank score < 580:
  - Wymondham: 506
  - Wigan & Orrel: 532
  - Clwyd: 545
  - Meriden Summer: 568
  - Exmouth: 569
  - Peacock: 576
